# QuantStream — Run 004: kiểm tra độ ổn định LoRA qua ba seed

## Mục tiêu

**Run 004 đã hoàn thành ba seed × 200 steps.** Xem [báo cáo kết quả](https://github.com/taiiswibu/quantstream-crypto-research/blob/research/run-004/docs/experiments/run-004-seed-stability.md); bản notebook này không nhúng các output Kaggle đã chạy. Không cần train thêm để hoàn thành vòng nghiên cứu.

Khi tái lập có chủ đích, **lần đầu giữ `MODE='preflight'` rồi Run All**. Dùng Dataset có đúng dữ liệu Run 003, bật Internet và GPU, chạy từ session mới. Không tạo lại dataset. Metadata export có thể khác; cell kiểm tra tính lại hash cả bốn Parquet và đối chiếu cấu hình/split/features với bản tham chiếu. Preflight chỉ thử một optimizer step trên chuỗi sin/cos, không train trên dữ liệu tài chính.

Notebook tải code mới theo Git SHA cố định; chỉ lấy snapshot từ Dataset cũ. Không dùng code cũ nằm trong input ZIP.

Sau khi preflight xong, tải `QuantStream_Run004_Preflight.zip` gửi lại để kiểm tra môi trường và khóa cấu hình trước ba lần fit. Nếu lỗi, vẫn gửi ZIP cùng traceback. ZIP xuất hiện chưa có nghĩa đã thành công.

Kế hoạch research: giữ LoRA B (200 steps, LR 1e-5), seeds 42/123/2026, context 128, batch 8 và cùng 512 cửa sổ development đã xem. Không chọn seed đẹp nhất và không promote model.

## 1. Cài thư viện và thử gắn adapter

Restart Session trước Run All. Giữ PyTorch/CUDA của Kaggle; gỡ optional torchao như Notebook 05.

In [ ]:
import subprocess,sys,os,importlib.util,importlib.metadata,platform
# Package discovery is cached after imports; changing pip packages cannot clear it safely.
loaded=[name for name in ('peft','torchao','chronos','transformers')
        if any(module==name or module.startswith(name+'.') for module in sys.modules)]
if loaded:
    raise RuntimeError('Restart Session rồi Run All trước khi cài thư viện: '+', '.join(loaded))
os.environ['HF_HUB_DISABLE_XET']='1'
# This lab uses ordinary float32 weights, not torchao quantization.
subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','torchao'])
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'chronos-forecasting==2.3.2','peft==0.21.2','accelerate>=1.1,<2',
    'datasets>=3,<5','pyarrow>=17,<27','scikit-learn>=1.5,<2'])
importlib.invalidate_caches()
if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao vẫn còn sau khi gỡ; kiểm tra pip output và Restart Session.')
import torch
from peft import LoraConfig,get_peft_model
from peft.import_utils import is_torchao_available
if is_torchao_available():
    raise RuntimeError('Lab yêu cầu PEFT chạy không có torchao.')
# Exercise the same adapter-injection path that previously failed, without a download.
smoke=get_peft_model(torch.nn.Sequential(torch.nn.Linear(4,4)),
                    LoraConfig(r=2,lora_alpha=4,target_modules=['0']))
smoke(torch.ones(1,4)).sum().backward()
if not any(p.requires_grad and p.grad is not None for p in smoke.parameters()):
    raise RuntimeError('LoRA preflight không tạo được gradient cho adapter.')
del smoke
LORA_ENVIRONMENT={'python':platform.python_version(),'torch':torch.__version__,
    'packages':{name:importlib.metadata.version(name) for name in
        ('chronos-forecasting','peft','transformers','accelerate','datasets')},
    'torchao_installed':False,'lora_injection_backward_ok':True}
LORA_PIP_FREEZE=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
print('LoRA preflight OK:',LORA_ENVIRONMENT)

## 2. Chọn chế độ

Giữ nguyên các tham số. `NOTEBOOK_URL` chỉ cần điền trước khi chuyển sang train sau khi preflight được kiểm tra. Thư mục output cũ được giữ để tránh âm thầm chạy lại trial lỗi.

In [ ]:
from pathlib import Path
import sys, subprocess, json, zipfile, shutil
from IPython.display import display, FileLink

MODE='preflight'
NOTEBOOK_URL=''  # Khi train: URL thật https://www.kaggle.com/code/<user>/<notebook>
SOURCE_COMMIT='bb47737b5bc400169a8bbd75f8c7534503ba18d5'
WORK=Path('/kaggle/working/quantstream_run004')
WORK.mkdir(parents=True,exist_ok=True)
CODE=WORK/('source-'+SOURCE_COMMIT[:12])
SNAPSHOT=WORK/'snapshot'
PREFLIGHT=WORK/'preflight'
OUTPUT=WORK/'run-004-seed-stability'
LOCK=PREFLIGHT/'candidate-lock.json'
if MODE not in {'preflight','train'}:
    raise ValueError('MODE phải là preflight hoặc train')
print('Mode:',MODE,'Source:',SOURCE_COMMIT)

## 3. Nạp đúng code và snapshot

Đối chiếu với manifest Run 003 gốc (`c8e01129…747fd0`). Chấp nhận metadata xuất khác khi hash của cả bốn Parquet và toàn bộ cấu hình/split/features giống bản gốc. Notebook tính lại hash từng file, giữ nguyên manifest đang dùng và ghi cả hai hash. Không đọc final holdout.

In [ ]:
if not CODE.exists():
    subprocess.check_call(['git','init','-q',str(CODE)])
    subprocess.check_call(['git','-C',str(CODE),'remote','add','origin',
                           'https://github.com/taiiswibu/quantstream-crypto-research.git'])
    subprocess.check_call(['git','-C',str(CODE),'fetch','-q','--depth','1','origin',SOURCE_COMMIT])
    subprocess.check_call(['git','-C',str(CODE),'checkout','-q','--detach','FETCH_HEAD'])
actual=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_COMMIT:
    raise ValueError('Source khác SHA đã khóa; giữ output để kiểm tra')
if subprocess.check_output(['git','-C',str(CODE),'status','--porcelain'],text=True).strip():
    raise ValueError('Source working tree phải sạch')
if any(name=='quantstream' or name.startswith('quantstream.') for name in sys.modules):
    raise RuntimeError('Restart Session để không dùng nhầm quantstream đã import')
sys.path.insert(0,str(CODE))

INPUT=Path('/kaggle/input')
archives=list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if len(archives)>1:
    raise ValueError('Chỉ Add Input một phiên bản dataset Run 003')
if not SNAPSHOT.exists():
    if archives:
        with zipfile.ZipFile(archives[0]) as archive:
            for info in archive.infolist():
                name=Path(info.filename)
                if name.is_absolute() or '..' in name.parts:
                    raise ValueError('Unsafe ZIP path')
                if name.parts and name.parts[0]=='snapshot':
                    archive.extract(info,WORK)
    else:
        manifests=[p for p in INPUT.rglob('manifest.json')
                   if (p.parent/'development.parquet').exists()]
        if len(manifests)!=1:
            raise FileNotFoundError('Add Input đúng dataset cũ đã chạy Run 003')
        shutil.copytree(manifests[0].parent,SNAPSHOT)
from quantstream.seed_lab import preflight_run004,run_seed_lab,archive_evidence,verify_run003_snapshot,SNAPSHOT_SHA
verified=verify_run003_snapshot(SNAPSHOT)
print('Snapshot:',verified['manifest_relation'])
print('Manifest hiện tại:',verified['manifest_sha256'])
print('Manifest Run 003:',verified['reference_manifest_sha256'])
print('Bốn file dữ liệu và cấu hình khớp Run 003. Chưa chạy research fit.')

## 4. Chạy và tải evidence

`preflight` tải checkpoint cố định và thử fit/predict với dữ liệu tổng hợp trên GPU. Lưu phiên bản Python/PyTorch/CUDA, package versions, cấu hình LoRA đầy đủ, source/data hashes và số optimizer step thật. Đây là bước kiểm tra môi trường, tách khỏi ngân sách 600 steps nghiên cứu.

Chỉ dùng `train` sau khi kiểm tra ZIP preflight và giữ nguyên lock/môi trường. Khi đó mỗi seed nạp lại pretrained model; ghi train loss mỗi step, trạng thái lỗi, dự báo và adapter riêng. Nếu phiên mất dữ liệu, gửi ZIP đã tải để được hướng dẫn khôi phục; không xóa trial rồi chạy lại nhằm lấy kết quả đẹp.

Nếu Kaggle bị kill cả process, `finally` có thể không chạy: Save Version/Download thư mục output còn lại. Preflight và ba fit của runner đã có evidence Kaggle được review; đây vẫn là bước kiểm tra cần thiết khi tái lập trên một môi trường mới.

In [ ]:
target=PREFLIGHT if MODE=='preflight' else OUTPUT
filename='QuantStream_Run004_Preflight.zip' if MODE=='preflight' else 'QuantStream_Run004_Output.zip'
archive=WORK.parent/filename
# Refuse before entering finally, so a repeated click cannot overwrite a previous archive.
if target.exists() or archive.exists():
    raise FileExistsError('Đã có evidence. Tải ZIP cũ và gửi kiểm tra, không xóa để chạy lại.')
try:
    if MODE=='preflight':
        result=preflight_run004(SNAPSHOT,PREFLIGHT)
        print('Preflight hoàn tất. Research fits đã chạy: 0')
        print('Tải QuantStream_Run004_Preflight.zip và gửi lại trước khi train.')
    else:
        result=run_seed_lab(SNAPSHOT,OUTPUT,LOCK,NOTEBOOK_URL)
        import pandas as pd
        display(pd.DataFrame(result['metrics']))
        print('Deployment:',result['deployment_status'])
finally:
    saved=archive_evidence(target,archive)
    if saved:
        display(FileLink(saved))

## 5. Đọc kết quả

Preflight thành công khi `status.json` ghi `completed`, có `candidate-lock.json`, smoke `global_step=1`, adapter và forecast hợp lệ. Chưa có kết quả tài chính nào ở giai đoạn này.

Research hoàn tất khi đủ ba seed, mỗi seed `global_step=200`, cùng 512 hàng/baseline và có `comparison.json`. MAE delta âm là giảm sai số trên development đã xem; không suy ra lợi nhuận. Báo mọi seed và từng tài sản, mean/std, paired bootstrap theo tuần UTC. Giữ `research_only_no_promotion` dù có seed tốt hơn zero-shot.